In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import uuid

# -----------------------------
# CONFIG
# -----------------------------
MAX_BOM_LEVELS = 3
MAX_CHILDREN = 3
SEED = 42

np.random.seed(SEED)

# -----------------------------
# LOAD EXISTING DATA
# -----------------------------
products_df = pd.read_csv("products.csv")
orders_df = pd.read_csv("orders_phase2.csv", parse_dates=[
    "order_date", "due_date", "release_time", "material_available_time"
])

# -----------------------------
# STEP 1: CREATE BOM
# -----------------------------
def generate_bom(products_df):
    bom_rows = []

    product_ids = products_df["product_id"].tolist()

    for parent in product_ids:
        num_children = np.random.randint(0, MAX_CHILDREN)

        children = np.random.choice(product_ids, num_children, replace=False)

        for child in children:
            if child == parent:
                continue

            bom_rows.append({
                "parent_product": parent,
                "child_product": child,
                "quantity": np.random.randint(1, 5)
            })

    return pd.DataFrame(bom_rows)

bom_df = generate_bom(products_df)

# -----------------------------
# STEP 2: EXPLODE ORDERS
# -----------------------------
order_counter = 0

def generate_order_id():
    global order_counter
    order_counter += 1
    return f"MO_{order_counter:05d}"


def explode_order(order_row, bom_df, level=0, parent_order_id=None):
    """
    Recursively explode order into sub-orders
    """
    exploded_orders = []
    links = []

    new_order_id = generate_order_id()

    # Create current order
    current_order = {
        "order_id": new_order_id,
        "product_id": order_row["product_id"],
        "quantity": order_row["quantity"],
        "order_date": order_row["order_date"],
        "due_date": order_row["due_date"],
        "release_time": order_row["release_time"],
        "material_available_time": order_row["material_available_time"],
        "penalty_per_hour": order_row["penalty_per_hour"],
        "level": level
    }

    exploded_orders.append(current_order)

    # Link to parent
    if parent_order_id:
        links.append({
            "parent_order_id": parent_order_id,
            "child_order_id": new_order_id
        })

    # Stop recursion
    if level >= MAX_BOM_LEVELS:
        return exploded_orders, links

    # Find children in BOM
    children = bom_df[bom_df["parent_product"] == order_row["product_id"]]

    for _, child in children.iterrows():
        child_order = order_row.copy()
        child_order["product_id"] = child["child_product"]
        child_order["quantity"] = order_row["quantity"] * child["quantity"]

        # Shift timing slightly earlier for components
        child_order["due_date"] = order_row["due_date"] - timedelta(hours=4 * (level + 1))
        child_order["material_available_time"] = order_row["order_date"]

        sub_orders, sub_links = explode_order(
            child_order,
            bom_df,
            level + 1,
            parent_order_id=new_order_id
        )

        exploded_orders.extend(sub_orders)
        links.extend(sub_links)

    return exploded_orders, links


# -----------------------------
# STEP 3: PROCESS ALL ORDERS
# -----------------------------
all_orders = []
all_links = []

for _, order in orders_df.iterrows():
    exploded, links = explode_order(order, bom_df)
    all_orders.extend(exploded)
    all_links.extend(links)

orders_multi_df = pd.DataFrame(all_orders)
links_df = pd.DataFrame(all_links)

# -----------------------------
# STEP 4: SAVE OUTPUT
# -----------------------------
bom_df.to_csv("bom.csv", index=False)
orders_multi_df.to_csv("orders_multilevel.csv", index=False)
links_df.to_csv("order_links.csv", index=False)

print("Phase 3 data generated:")
print(" - bom.csv")
print(" - orders_multilevel.csv")
print(" - order_links.csv")

Phase 3 data generated:
 - bom.csv
 - orders_multilevel.csv
 - order_links.csv
